# Notebook 02 — Load a knowledge base into Pinecone

**VidyaSankalp · Applied GenAI Engineering** · Tutor: Prudhvi Akella

Do **Notebook 01** first: it explains embeddings, the index and namespaces. This notebook reuses the same index
`ecom-kb-tutorial`, and loads a whole spreadsheet into a new namespace, `ecom-kb`.

Needs the same `.env` (OpenAI + Pinecone keys) and `kb_feed_ecom.xlsx` in this folder.

In [ ]:
import json
import os
import time

import openpyxl
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

load_dotenv()

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"
DIMENSION = 1536
INDEX_NAME = "ecom-kb-tutorial"


def embed(texts):
    # Send a list of texts to OpenAI, get back one list of 1536 numbers per text.
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]


pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
if not pc.has_index(INDEX_NAME):                    # in case Notebook 01 was skipped or cleaned up
    pc.create_index(name=INDEX_NAME, dimension=DIMENSION, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
index = pc.Index(INDEX_NAME)


def wait_for(namespace, expected, seconds=60):
    # New records take a few seconds to show up in counts and searches.
    for _ in range(seconds // 2):
        found = index.describe_index_stats().namespaces.get(namespace)
        count = found.vector_count if found else 0
        if count >= expected:
            print(f"{namespace}: {count} records ready")
            return
        time.sleep(2)
    print(f"{namespace}: still {count} after {seconds} s — run this cell again")


print("Connected to", INDEX_NAME)

---
# Part 2 — load a real knowledge base

`kb_feed_ecom.xlsx` describes an e-commerce database for an AI assistant that answers questions with SQL.

| Tab | Rows | One row is |
|---|---|---|
| `nlq_table` | 14 | a table and what it means |
| `nlq_column` | 185 | a column: meaning, synonyms, allowed values |
| `nlq_join` | 16 | how two tables connect |
| `glossary` | 66 | a business word → formula → columns |
| `nlq_examples` | 61 | a question and the SQL that answers it |
| `capability_card` | 3 | what the assistant can and cannot answer |

```
 for each tab:  read rows ──► one text + id + metadata per row ──► embed (100 at a time) ──► upsert, namespace "ecom-kb"
```

## 7. Open the workbook and look at the tabs

In [ ]:
workbook = openpyxl.load_workbook("kb_feed_ecom.xlsx", read_only=True)

for sheet in workbook.worksheets:
    print(f"{sheet.title:16} {sheet.max_row - 1:>4} rows")

## 8. Read one tab into a list of dicts

Row 1 holds the column names. Every other row becomes a dict: `{column name: value}` — the same shape as a database row on Python Day 01.

In [ ]:
def read_sheet(name):
    rows = list(workbook[name].iter_rows(values_only=True))
    header = rows[0]
    records = []
    for values in rows[1:]:
        if all(v is None for v in values):          # skip empty lines at the bottom
            continue
        records.append(dict(zip(header, values)))
    return records


columns = read_sheet("nlq_column")
print("Rows:", len(columns))
print(json.dumps(columns[0], indent=2, default=str)[:900])

## 9. Turn each row into one text, one id and metadata

A row has many fields. We choose **what to embed** (the meaning: description, synonyms) and **what to keep as metadata**
(labels to filter on: tab, table, data type — plus the text, so we can read it back).

```
 row from nlq_column                                   record for Pinecone
 ─────────────────────                                 ───────────────────
 table: ecom.orders                                    id:       column::ecom.orders.total_amount
 column: total_amount                       ──►        text  →   "Column ecom.orders.total_amount (FLOAT).
 description: Merchandise subtotal …                   (embedded) Merchandise subtotal … Also called: Revenue; Sales …"
 analyst_synonyms: Order Total; Revenue …              metadata: {sheet: nlq_column, table: ecom.orders,
 data_type: FLOAT                                                 column: total_amount, data_type: FLOAT, text: …}
```

Pinecone metadata values must be text, numbers, True/False, or lists of text — **never empty (None)**. `metadata_for` only adds fields that have a value.

In [ ]:
def text_for(sheet, row):
    if sheet == "nlq_table":
        return row["description"]                                    # already a full Markdown description
    if sheet == "nlq_column":
        text = f"Column {row['table']}.{row['column']} ({row['data_type']}). {row['description']}"
        if row["analyst_synonyms"]:
            text += f" Also called: {row['analyst_synonyms']}."
        if row["enum_values"]:
            text += f" Values: {row['enum_values']}."
        return text
    if sheet == "nlq_join":
        return (f"Join {row['left_table']} to {row['right_table']} on {row['join_on']} "
                f"({row['cardinality']}). {row['description']}")
    if sheet == "glossary":
        return (f"{row['term']} ({row['category']}): {row['meaning']} "
                f"Also called: {row['synonyms']}. Columns: {row['maps_to']}")
    if sheet == "nlq_examples":
        return f"{row['question']} Also asked as: {row['paraphrases']}"   # embed the question, not the SQL
    if sheet == "capability_card":
        return row["description"]


def id_for(sheet, row, number):
    if sheet == "nlq_table":
        return f"table::{row['table']}"
    if sheet == "nlq_column":
        return f"column::{row['table']}.{row['column']}"
    if sheet == "nlq_join":
        return f"join::{row['left_table']}->{row['right_table']}::{row['join_on']}"
    if sheet == "glossary":
        return f"glossary::{row['term']}"
    return f"{sheet}::{number}"                                       # examples, capability card


def metadata_for(sheet, row, text):
    labels = {"sheet": sheet, "text": text}
    for field in ("table", "column", "data_type", "category", "pattern", "tables_used"):
        if row.get(field) is not None:
            labels[field] = str(row[field])
    if sheet == "nlq_examples":
        labels["sql"] = row["query"]                                  # keep the SQL to show later
    return labels


row = columns[0]
text = text_for("nlq_column", row)
print("id      :", id_for("nlq_column", row, 0))
print("text    :", text)
print("metadata:", metadata_for("nlq_column", row, text))

**Why the SQL is metadata, not embedded:** people search with questions, so we embed the *question*.
The SQL rides along as metadata and comes back with the match.

**Not everything should be embedded.** Rows marked `embed_exempt = Y` (map coordinates like `lat`, `lon`)
have no meaning a search could use, so we skip them.

## 10. Read every tab and build the records

In [ ]:
SHEETS = ["nlq_table", "nlq_column", "nlq_join", "glossary", "nlq_examples", "capability_card"]

kb_records = []                                  # one dict per row: id, text, metadata
for sheet in SHEETS:
    kept, skipped = 0, 0
    for number, row in enumerate(read_sheet(sheet), start=1):
        if row.get("embed_exempt") in ("Y", True):
            skipped += 1
            continue
        text = text_for(sheet, row)
        kb_records.append({
            "id": id_for(sheet, row, number),
            "text": text,
            "metadata": metadata_for(sheet, row, text),
        })
        kept += 1
    print(f"{sheet:16} kept {kept:>3}   skipped {skipped}")

print("\nTotal records:", len(kb_records))

## 11. Embed in batches and upsert

OpenAI accepts many texts in one call. Sending 100 at a time is far faster than 337 separate calls.

```
 kb_records[0:100]   ──embed──► upsert (namespace "ecom-kb")
 kb_records[100:200] ──embed──► upsert
 kb_records[200:300] ──embed──► upsert
 kb_records[300:337] ──embed──► upsert
```

In [ ]:
KB = "ecom-kb"                                    # the namespace for Part 2
BATCH = 100

for start in range(0, len(kb_records), BATCH):
    batch = kb_records[start:start + BATCH]
    values = embed([r["text"] for r in batch])
    index.upsert(
        vectors=[{"id": r["id"], "values": v, "metadata": r["metadata"]} for r, v in zip(batch, values)],
        namespace=KB,
    )
    print(f"upserted {start + 1}–{start + len(batch)}")

wait_for(KB, len(kb_records))

## 12. Search the whole knowledge base

One helper: embed the question, ask for the closest records, print them.

In [ ]:
def search(question, top_k=5, filter=None):
    result = index.query(vector=embed([question])[0], top_k=top_k, namespace=KB,
                         filter=filter, include_metadata=True)
    print(f"Q: {question}")
    for match in result.matches:
        print(f"   {match.score:.3f}  {match.id:48}  {match.metadata['text'][:70]}")
    print()
    return result


search("What was our GMV last month?")
search("Which deliveries arrived late?")
search("Which customers send back the most products?")

Look at the first question. In the whole workbook, the word **GMV** appears in exactly one place: the synonyms of the
glossary term *Revenue*. Is `glossary::Revenue` in your results? Which columns come with it?
For "send back", look for **Returned** — the workbook calls it returns and refunds, never "send back".

## 13. Narrow the search with metadata filters

| Filter | Means |
|---|---|
| `{"sheet": {"$eq": "glossary"}}` | only glossary terms |
| `{"$and": [{"sheet": {"$eq": "nlq_column"}}, {"table": {"$eq": "ecom.shipments"}}]}` | only columns of one table |
| `{"sheet": {"$in": ["nlq_table", "nlq_join"]}}` | tables and joins only |

In [ ]:
search("money we refunded", filter={"sheet": {"$eq": "glossary"}})

search("how long did it take",
       filter={"$and": [{"sheet": {"$eq": "nlq_column"}}, {"table": {"$eq": "ecom.shipments"}}]})

search("how do orders connect to customers", top_k=3, filter={"sheet": {"$in": ["nlq_table", "nlq_join"]}})

## 14. Find a similar question — and reuse its SQL

This is how an AI assistant uses the knowledge base: find an example question that means the same as the user's,
and hand its SQL to the LLM as a guide (**few-shot example**).

```
 "best selling brands this year" ──search, only nlq_examples──► closest example question
                                                                     │ metadata["sql"]
                                                                     ▼
                                                               its SQL query
```

In [ ]:
result = search("best selling brands this year", top_k=1, filter={"sheet": {"$eq": "nlq_examples"}})

best = result.matches[0].metadata
print("Tables used:", best["tables_used"])
print(best["sql"])

---
# Part 3 — it lives in the cloud

The records are stored in your Pinecone account, not on your laptop. Close the notebook, open it tomorrow, reconnect —
no embedding calls, no cost. You can also see the index at **app.pinecone.io → Indexes**.

In [ ]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index(INDEX_NAME)

stats = index.describe_index_stats()
print("Dimension:", stats.dimension)
for name, namespace in stats.namespaces.items():
    print(f"  {name:15} {namespace.vector_count:>4} records")

### Fetch and delete by id

Ids let you read or remove one record directly — this is how you keep the index in sync when a row is changed
or deleted in the workbook (`action = UPDATE / DELETE`).

In [ ]:
found = index.fetch(ids=["glossary::Revenue"], namespace=KB)
print(found.vectors["glossary::Revenue"].metadata["text"][:200])

index.delete(ids=["capability_card::3"], namespace=KB)   # remove one record by its id
print("deleted capability_card::3 — run section 10 + 11 again to put it back")

### Clean up (optional)

The free plan allows only a few indexes. When the class is over, delete the tutorial index — **this removes every record in it.**

In [ ]:
# pc.delete_index(INDEX_NAME)      # remove the # and run to delete the whole index

## Recap

| Step | Code |
|---|---|
| Text → numbers | `client.embeddings.create(model=..., input=[...])` |
| Create an index | `pc.create_index(name=..., dimension=1536, metric="cosine", spec=ServerlessSpec(...))` |
| Insert / replace | `index.upsert(vectors=[{"id": ..., "values": [...], "metadata": {...}}], namespace=...)` |
| Search | `index.query(vector=[...], top_k=..., namespace=..., filter=..., include_metadata=True)` |
| Read / remove by id | `index.fetch(ids=[...])`, `index.delete(ids=[...])` |
| Read a spreadsheet | `openpyxl.load_workbook(...)` → `iter_rows(values_only=True)` → `dict(zip(header, row))` |

**Try it yourself**

1. Search for `"coupon"` with and without `filter={"sheet": {"$eq": "glossary"}}`. What changes?
2. Ask a question the assistant can't answer (e.g. `"website visitors per day"`) — which record comes first? (Hint: `capability_card`.)
3. Change `top_k` to 10 for "late deliveries". At what score do the results stop being useful?
4. Upsert one record of your own into `first-example` and find it with a question that shares **no** words with it.